Catboost history log

In [1]:
from types import SimpleNamespace
import mlflow
import mlflow.catboost
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split



In [2]:
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

In [11]:
# Загружаем данные
X = data
y = target


In [10]:
mlflow.set_tracking_uri("http://localhost:5801")
mlflow.set_experiment("Catboost test regress")


<Experiment: artifact_location='mlflow-artifacts:/15', creation_time=1735829699624, experiment_id='15', last_update_time=1735829699624, lifecycle_stage='active', name='Catboost test regress', tags={}>

In [12]:
# Разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Подготовка Pool для CatBoost
train_pool = Pool(X_train, y_train)
test_pool = Pool(X_test, y_test)


In [13]:
# Конфигурация CatBoost
params = {
    "iterations": 100,
    "learning_rate": 0.1,
    "depth": 6,
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "verbose": 10,
}

In [ ]:
# Начинаем эксперимент в MLflow
with mlflow.start_run():
    # Логируем параметры модели
    mlflow.log_params(params)

    # Инициализация модели
    model = CatBoostRegressor(**params)

    # Создаем коллбэк
    logger = MLflowLogger()

    # Обучение модели с логированием метрик
    _ = model.fit(
        train_pool,
        eval_set=test_pool,
        logging_level="Silent",
        callbacks=[logger],
    )

    # Оценка модели
    y_pred = model.predict(X_test)
    rmse_test = np.sqrt(np.mean((y_test - y_pred) ** 2))
    mlflow.log_metric("test_rmse", rmse_test)

    # Сохранение модели в MLflow
    mlflow.catboost.log_model(model, artifact_path="model")

print("Модель сохранена и метрики залогированы в MLflow.")

In [14]:
model = CatBoostRegressor(**params)

In [17]:
_ = model.fit(
    train_pool,
    eval_set=test_pool,
    logging_level='Info',
)


12, bin=112 score 127.0055841
5, bin=201 score 157.5072283
10, bin=35 score 159.5458341
6, bin=225 score 169.3323479
1, bin=5 score 177.0413864
9, bin=1 score 173.6565436
0:	learn: 8.7281973	test: 8.1557700	best: 8.1557700 (0)	total: 2.24ms	remaining: 222ms

5, bin=209 score 116.6295173
10, bin=32 score 139.8036706
0, bin=109 score 151.2464024
8, bin=7 score 139.088295
5, bin=19 score 145.9028099
0, bin=207 score 172.8591605

12, bin=37 score 102.9593689
12, bin=151 score 138.7730669
5, bin=232 score 142.4487346
5, bin=242 score 151.5543504
9, bin=55 score 153.0664817
4, bin=76 score 156.6777575

5, bin=218 score 99.50198551
12, bin=177 score 122.9492727
9, bin=27 score 130.4379924
4, bin=29 score 139.6025145
8, bin=7 score 128.4981215
3, bin=0 score 135.1450738

12, bin=54 score 96.00104083
7, bin=58 score 110.3580605
12, bin=163 score 132.4156953
2, bin=64 score 125.7134346
10, bin=28 score 125.7699914
0, bin=238 score 130.9000696

12, bin=64 score 89.52642193
5, bin=221 score 99.85